In [ ]:
# [0] 준비: 한글 폰트 + 1~5단계 결과 표 불러오기 (무거운 전처리는 하지 않음)
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams.update({'axes.titlesize': 17, 'axes.labelsize': 14, 'xtick.labelsize': 13,
                     'ytick.labelsize': 12, 'legend.fontsize': 13, 'figure.titlesize': 19})

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TABLES = ROOT / 'outputs' / 'tables'
COLOR = {2019: '#2a78d6', 2026: '#eb6834'}
YEARS = (2019, 2026)


def style_axis(ax):
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', color='#e5e5e5', lw=0.8)
    ax.set_axisbelow(True)


bike_monthly = pd.read_csv(TABLES / '00_bike_monthly_daily_avg.csv')
overview = pd.read_csv(TABLES / '02_overview_summary.csv')
trip_types = pd.read_csv(TABLES / '02_trip_type_share.csv')
stations = pd.read_csv(TABLES / '01_analysis_stations.csv')
bike_assign = pd.read_csv(TABLES / '01_bike_station_assignment.csv')
hyp1_growth = pd.read_csv(TABLES / '03_hyp1_station_growth.csv')
hyp1_bar = pd.read_csv(TABLES / '03_hyp1_group_bar.csv')
hyp1_top10 = pd.read_csv(TABLES / '03_hyp1_top10_pairs.csv')
hyp2_hourly = pd.read_csv(TABLES / '04_hyp2_hourly.csv')
hyp2_bar = pd.read_csv(TABLES / '04_hyp2_share_bar.csv')
hyp2_growth = pd.read_csv(TABLES / '04_hyp2_station_growth.csv')
robust = pd.read_csv(TABLES / '05_robustness_comparison.csv')
robust_p = pd.read_csv(TABLES / '05_robustness_p_detail.csv')
tests = pd.read_csv(TABLES / 'tests_summary.csv')

In [ ]:
# [1] 00_bike_monthly — 따릉이 1~6월 월별 하루 평균 대여건수, 2019 vs 2026 (분석 기간 5~6월 선정 근거)
fig, ax = plt.subplots(figsize=(12, 6))
width = 0.38
for i, year in enumerate(YEARS):
    d = bike_monthly[bike_monthly.year == year]
    ax.bar(d.month + (i - 0.5) * width, d.daily_avg, width=width - 0.03, color=COLOR[year], label=f'{year}년')
    for m, v in zip(d.month, d.daily_avg):
        if m in (5, 6):
            ax.text(m + (i - 0.5) * width, v, f'{v:,.0f}', ha='center', va='bottom', fontsize=11, color='#333333')
ax.axvspan(4.5, 6.5, color='#999999', alpha=0.12, lw=0)
ax.text(5.5, ax.get_ylim()[1] * 0.98, '분석 기간 (5~6월)', ha='center', va='top', fontsize=13, color='#555555')
ax.set_xticks(range(1, 7), [f'{m}월' for m in range(1, 7)])
ax.set_ylabel('하루 평균 대여건수')
ax.set_title('따릉이 월별 하루 평균 대여건수 (1~6월)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
style_axis(ax)
ax.legend(frameon=False, loc='upper left')
fig.tight_layout()
plt.show()

In [ ]:
# [2-1] 01_map (서울 전체) — 분석 대상 역, 역 반경 500m, 배정된 따릉이 대여소 (배경: CartoDB Positron)
import warnings
from matplotlib.patches import Circle
from matplotlib.lines import Line2D

R_EARTH = 6378137.0


def to_mercator(lon, lat):
    """경위도 → EPSG:3857 (m)."""
    lon, lat = np.radians(np.asarray(lon, dtype=float)), np.radians(np.asarray(lat, dtype=float))
    return R_EARTH * lon, R_EARTH * np.log(np.tan(np.pi / 4 + lat / 2))


def mercator_len(meters, lat):
    """실제 거리(m) → 해당 위도에서의 EPSG:3857 길이 (위도가 높을수록 늘어남)."""
    return meters / np.cos(np.radians(lat))


st_x, st_y = to_mercator(stations.lon, stations.lat)
bk_x, bk_y = to_mercator(bike_assign.lon, bike_assign.lat)
assigned_mask = bike_assign.station.notna().values


def add_basemap(ax, zoom):
    """회색 배경 지도. CARTO_API_KEY가 있으면 CartoDB Positron, 없으면 키가 필요 없는 Esri 회색 지도.
    (CartoDB는 2025년부터 API 키 필요 — 키 없이 요청하면 워터마크 타일이 옴. Esri는 한국 구간 줌 13까지 제공)"""
    import os
    try:
        import contextily as ctx
        key = os.environ.get('CARTO_API_KEY')
        if key:
            source, zoom_use = ctx.providers.CartoDB.Positron(apikey=key), zoom + 1
        else:
            source, zoom_use = ctx.providers.Esri.WorldGrayCanvas, min(zoom, 13)
        ctx.add_basemap(ax, source=source, crs='EPSG:3857', zoom=zoom_use, attribution_size=7)
    except Exception as e:  # 인터넷·타일 서버 문제 → 배경 없이 그림
        warnings.warn(f'배경 지도를 불러오지 못해 배경 없이 그립니다: {type(e).__name__}: {e}')


def draw_map(ax, extent=None, label_stations=False):
    zoom = extent is not None
    if zoom:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
    else:
        pad = 1500
        ax.set_xlim(min(st_x.min(), bk_x.min()) - pad, max(st_x.max(), bk_x.max()) + pad)
        ax.set_ylim(min(st_y.min(), bk_y.min()) - pad, max(st_y.max(), bk_y.max()) + pad)
    ax.set_aspect('equal')
    add_basemap(ax, zoom=13 if zoom else 11)
    ax.scatter(bk_x[~assigned_mask], bk_y[~assigned_mask], s=18 if zoom else 4, color='#9a9a9a', lw=0, zorder=2)
    ax.scatter(bk_x[assigned_mask], bk_y[assigned_mask], s=22 if zoom else 6, color='#1baf7a', lw=0, zorder=3)
    for x, y, lat in zip(st_x, st_y, stations.lat):
        ax.add_patch(Circle((x, y), mercator_len(500, lat), fill=False, ec='#2a78d6',
                            lw=1.2 if zoom else 0.6, alpha=0.85, zorder=4))
    ax.scatter(st_x, st_y, s=30 if zoom else 6, color='#222222', zorder=5)
    if label_stations:
        for x, y, lat, name in zip(st_x, st_y, stations.lat, stations.station):
            if extent[0] < x < extent[1] and extent[2] < y < extent[3]:
                ax.text(x, y + mercator_len(120, lat), name, fontsize=11, ha='center', va='bottom', zorder=6,
                        bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.7))
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_color('#bbbbbb')


legend_items = [Line2D([], [], marker='o', ls='', color='#222222', label=f'분석 대상 지하철역 ({len(stations)}곳)'),
                Line2D([], [], marker='o', ls='', mfc='none', mec='#2a78d6', ms=12, label='역 반경 500m'),
                Line2D([], [], marker='o', ls='', color='#1baf7a', label='역에 배정된 따릉이 대여소'),
                Line2D([], [], marker='o', ls='', color='#9a9a9a', label='배정 안 된 대여소 (역에서 500m 밖)')]

fig, ax = plt.subplots(figsize=(13, 10))
draw_map(ax)
ax.legend(handles=legend_items, loc='lower left', frameon=True, framealpha=0.9)
ax.set_title('분석 대상 지하철역과 따릉이 대여소 (대여소 위치: 26.6월 기준, 없으면 21.1월)')
fig.tight_layout()
plt.show()

In [ ]:
# [2-2] 01_map (확대) — 역이 가장 밀집한 구역 3km × 3km (반경 1.5km 안 역 수가 가장 많은 역 중심)
d = np.hypot((stations.lat.values[:, None] - stations.lat.values) * 111,
             (stations.lon.values[:, None] - stations.lon.values) * 111 * np.cos(np.radians(stations.lat.mean())))
c = (d <= 1.5).sum(axis=1).argmax()
center = stations.iloc[c]
half = mercator_len(1500, center.lat)
extent = (st_x[c] - half, st_x[c] + half, st_y[c] - half, st_y[c] + half)
fig, ax = plt.subplots(figsize=(12, 12.6))
draw_map(ax, extent=extent, label_stations=True)
ax.legend(handles=legend_items, loc='upper center', bbox_to_anchor=(0.5, -0.01), ncol=2, frameon=False)
ax.set_title(f'확대: {center.station} 주변 3km × 3km')
fig.text(0.5, 0.012, '대여소는 반경 500m 안에서 가장 가까운 역 하나에만 배정 (원이 겹쳐도 중복 배정 없음)',
         ha='center', fontsize=12, color='#666666')
fig.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()

In [ ]:
# [3] 02_overview — 2019 vs 2026 지하철·따릉이 평일/주말 하루 평균 (에러바: 95% 신뢰구간, 단위: 날짜)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
panels = [('지하철 (분석 대상 295개 역)', '하루 승하차 (만 명)', 1e4),
          ('따릉이 (서울 전체)', '하루 이동 (건)', 1)]
width = 0.36
for ax, (measure, ylabel, scale) in zip(axes, panels):
    d = overview[overview.measure == measure]
    for i, year in enumerate(YEARS):
        dy = d[d.year == year].set_index('day_type').loc[['평일', '주말']]
        x = [0 + (i - 0.5) * width, 1 + (i - 0.5) * width]
        err = [(dy['mean'] - dy.ci_low) / scale, (dy.ci_high - dy['mean']) / scale]
        ax.bar(x, dy['mean'] / scale, width=width - 0.03, color=COLOR[year], label=f'{year}년',
               yerr=err, capsize=5, error_kw={'ecolor': '#444444', 'lw': 1.2})
    for j, day_type in enumerate(['평일', '주말']):
        r = d[d.day_type == day_type].set_index('year')
        top = r.ci_high.max() / scale
        ax.text(j, top * 1.03, f'{r.change_pct.iloc[0]:+.1f}%', ha='center', va='bottom', fontsize=14,
                fontweight='bold', color='#333333')
    ax.set_xticks([0, 1], [f'평일 (n=40 / 39일)', '주말 (n=18 / 18일)'])
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, d.ci_high.max() / scale * 1.18)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
    ax.set_title(measure)
    style_axis(ax)
axes[0].legend(frameon=False, loc='upper right')
fig.suptitle('2019 → 2026 (5~6월) 하루 평균 변화: 지하철은 약 3% 감소, 따릉이는 70~88% 증가')
fig.tight_layout()
plt.show()

In [ ]:
# [4] 03_trip_types — 따릉이 이동 유형별 비중 변화, 2019 vs 2026 (평일 / 주말)
order = ['따릉이 역간 단거리 이동', '따릉이 역 연계 이동', '따릉이 기타 이동']
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
width = 0.36
for ax, day_type in zip(axes, ['평일', '주말']):
    d = trip_types[trip_types.day_type == day_type]
    for i, year in enumerate(YEARS):
        dy = d[d.year == year].set_index('trip_type').loc[order]
        x = [k + (i - 0.5) * width for k in range(len(order))]
        ax.bar(x, dy.share_pct, width=width - 0.03, color=COLOR[year], label=f'{year}년')
        for xi, v in zip(x, dy.share_pct):
            ax.text(xi, v + 0.6, f'{v:.1f}%', ha='center', va='bottom', fontsize=12, color='#333333')
    ax.set_xticks(range(len(order)), [o.replace('따릉이 ', '').replace(' 이동', '') + '\n이동' for o in order])
    ax.set_title(day_type)
    style_axis(ax)
axes[0].set_ylabel('전체 따릉이 이동 중 비중 (%)')
axes[0].set_ylim(0, 55)
axes[0].legend(frameon=False, loc='upper right')
fig.suptitle('따릉이 이동 유형별 비중: 역 연계 이동 비중 증가, 역간 단거리 이동 비중 소폭 감소')
fig.tight_layout()
plt.show()

In [ ]:
# [5] 04_hyp1_scatter — 역별 따릉이 역간 단거리 이동 증가율 vs 지하철 증가율 (평일 하루 평균, ①-1 Spearman)
t = tests.set_index('검정ID').loc['①-1']
g = hyp1_growth
fig, ax = plt.subplots(figsize=(12, 8))
ax.axhline(0, color='#999999', lw=1)
ax.axvline(0, color='#999999', lw=1)
ax.scatter(g.hop_growth_pct, g.subway_growth_pct, s=40, color='#2a78d6', alpha=0.6, ec='white', lw=0.6)
labeled = pd.concat([g.nlargest(4, 'subway_growth_pct'), g.nlargest(2, 'hop_growth_pct')]).drop_duplicates('station')
for _, r in labeled.iterrows():
    ax.annotate(r.station, (r.hop_growth_pct, r.subway_growth_pct), xytext=(6, 4), textcoords='offset points',
                fontsize=11, color='#333333')
ax.text(0.98, 0.97, f"Spearman {t['통계량'].split(' (')[0]}, {t['p<0.05']}\n역 n = {t.n}",
        transform=ax.transAxes, ha='right', va='top', fontsize=14,
        bbox=dict(boxstyle='round', fc='white', ec='#cccccc'))
ax.set_xlabel('따릉이 역간 단거리 이동 증가율 (%)')
ax.set_ylabel('지하철 이용 증가율 (%)')
ax.set_title('따릉이 역간 단거리 이동 증가율 vs 지하철 증가율 (2019→2026, 5~6월 평일)')
style_axis(ax)
fig.tight_layout()
plt.show()

In [ ]:
# [6] 05_hyp1_bar — 역간 단거리 이동 많이 는 역 vs 적게 는 역의 지하철 증가율 중앙값 (①-2 순위합 검정(W))
t = tests.set_index('검정ID').loc['①-2']
b = hyp1_bar.set_index('group').loc[['많이 는 역', '적게 는 역']]
fig, ax = plt.subplots(figsize=(9, 7))
x = [0, 1]
ax.bar(x, b.median_subway_growth_pct, width=0.55, color=['#1baf7a', '#9aa0a6'],
       yerr=[b.median_subway_growth_pct - b.ci_low, b.ci_high - b.median_subway_growth_pct],
       capsize=8, error_kw={'ecolor': '#444444', 'lw': 1.4})
ax.axhline(0, color='#555555', lw=1)
for xi, (_, r) in zip(x, b.iterrows()):
    up = r.median_subway_growth_pct >= 0
    ax.text(xi + 0.3, r.median_subway_growth_pct, f'{r.median_subway_growth_pct:+.1f}%',
            ha='left', va='center', fontsize=15, fontweight='bold', color='#333333')
stars = b.stars.iloc[0] if isinstance(b.stars.iloc[0], str) else ''
if stars:
    top = b.ci_high.max() + 1.5
    ax.plot([0, 0, 1, 1], [top - 0.4, top, top, top - 0.4], color='#333333', lw=1.2)
    ax.text(0.5, top + 0.1, stars, ha='center', va='bottom', fontsize=18)
ax.set_xticks(x, [f'역간 단거리 이동\n많이 는 역 (n={b.n.iloc[0]})', f'역간 단거리 이동\n적게 는 역 (n={b.n.iloc[1]})'])
ax.set_ylabel('지하철 이용 증가율 중앙값 (%)')
ax.set_ylim(min(b.ci_low.min() - 2, -1), max(b.ci_high.max() + 4, 1))
ax.set_title(f"평일 지하철 이용 증가율 (순위합 검정(W), {t['p<0.05']})")
ax.text(0.99, 0.02, '에러바: 중앙값의 95% 부트스트랩 신뢰구간 / *p<0.05, **p<0.01', transform=ax.transAxes,
        ha='right', fontsize=10, color='#666666')
style_axis(ax)
fig.tight_layout()
plt.show()

In [ ]:
# [7] 06_hyp2_hourly — 평일 시간대별 따릉이 반납 분포: 역 근처 vs 역에서 먼 반납, 2019 vs 2026 (반납일시 기준)
fig, ax = plt.subplots(figsize=(13, 7))
ax.axvspan(6.5, 8.5, color='#999999', alpha=0.12, lw=0)
ax.text(7.5, 0.5, '출근 시간\n7~9시', ha='center', va='bottom', fontsize=12, color='#555555')
for year in YEARS:
    for where, ls in (('역 근처 반납', '-'), ('역에서 먼 반납', '--')):
        d = hyp2_hourly[(hyp2_hourly.year == year) & (hyp2_hourly['where'] == where)].sort_values('hour')
        ax.plot(d.hour, d.share_pct, ls=ls, lw=2.2, color=COLOR[year], marker='o', ms=5, label=f'{year}년 {where}')
ax.set_xticks(range(0, 24, 2), [f'{h}시' for h in range(0, 24, 2)])
ax.set_xlim(-0.5, 23.5)
ax.set_ylim(0, None)
ax.set_xlabel('반납 시각')
ax.set_ylabel('하루 반납 중 비중 (%)')
ax.set_title('평일 시간대별 따릉이 반납 분포 (2019 vs 2026, 5~6월)')
style_axis(ax)
ax.legend(frameon=False, loc='upper left', ncol=2)
fig.tight_layout()
plt.show()

In [ ]:
# [8] 07_top10 — 따릉이 역간 단거리 이동이 많은 역 구간 Top 10 (2026 평일 하루 평균, 2019과 비교)
d = hyp1_top10.iloc[::-1]
fig, ax = plt.subplots(figsize=(12, 8))
y = np.arange(len(d))
h = 0.38
ax.barh(y - h / 2, d.wd_2019, height=h - 0.03, color=COLOR[2019], label='2019년')
ax.barh(y + h / 2, d.wd_2026, height=h - 0.03, color=COLOR[2026], label='2026년')
for yi, v, gr in zip(y, d.wd_2026, d.growth_pct):
    ax.text(v + 8, yi + h / 2, f'{v:,.0f}건 ({gr:+.0f}%)', va='center', fontsize=12, color='#333333')
ax.set_yticks(y, d.pair)
ax.set_xlabel('평일 하루 평균 이동 (건, 양방향 합)')
ax.set_xlim(0, d.wd_2026.max() * 1.25)
ax.set_title('따릉이 역간 단거리 이동이 많은 역 구간 Top 10 (2026 평일 기준)')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', color='#e5e5e5', lw=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc='lower right')
fig.tight_layout()
plt.show()

In [ ]:
# [9] 08_hyp2_share_bar — 평일 7~9시 역 근처 반납 비중, 2019 vs 2026 (②-1 순위합 검정(W), 단위: 평일 하루)
t = tests.set_index('검정ID').loc['②-1']
b = hyp2_bar.set_index('year').loc[list(YEARS)]
fig, ax = plt.subplots(figsize=(9, 7))
x = [0, 1]
ax.bar(x, b.median_share_pct, width=0.55, color=[COLOR[y] for y in YEARS],
       yerr=[b.median_share_pct - b.ci_low, b.ci_high - b.median_share_pct],
       capsize=8, error_kw={'ecolor': '#444444', 'lw': 1.4})
for xi, (_, r) in zip(x, b.iterrows()):
    ax.text(xi, r.ci_high + 0.3, f'{r.median_share_pct:.1f}%\n(하루 {r.morning_returns_daily_avg:,.0f}건)',
            ha='center', va='bottom', fontsize=14, color='#333333')
stars = b.stars.iloc[0] if isinstance(b.stars.iloc[0], str) else ''
if stars:
    top = b.ci_high.max() + 3.2
    ax.plot([0, 0, 1, 1], [top - 0.4, top, top, top - 0.4], color='#333333', lw=1.2)
    ax.text(0.5, top + 0.1, stars, ha='center', va='bottom', fontsize=18)
ax.set_xticks(x, [f'{y}년 (평일 n={n}일)' for y, n in zip(YEARS, b.n_days)])
ax.set_ylabel('역 근처 반납 중 7~9시 비중 (%, 하루 값의 중앙값)')
ax.set_ylim(0, b.ci_high.max() + 6)
ax.set_title(f"평일 7~9시 역 근처 반납 비중 (순위합 검정(W), {t['p<0.05']})")
ax.text(0.99, 0.02, '에러바: 중앙값의 95% 부트스트랩 신뢰구간 / *p<0.05, **p<0.01', transform=ax.transAxes,
        ha='right', fontsize=10, color='#666666')
style_axis(ax)
fig.tight_layout()
plt.show()

In [ ]:
# [10] 09_hyp2_scatter — 역별 따릉이 역 연계 이동 증가율 vs 지하철 증가율 (평일 하루 평균, ②-2 Spearman)
t = tests.set_index('검정ID').loc['②-2']
g = hyp2_growth
fig, ax = plt.subplots(figsize=(12, 8))
ax.axhline(0, color='#999999', lw=1)
ax.axvline(0, color='#999999', lw=1)
ax.scatter(g.link_growth_pct, g.subway_growth_pct, s=40, color='#2a78d6', alpha=0.6, ec='white', lw=0.6)
labeled = pd.concat([g.nlargest(4, 'subway_growth_pct'), g.nlargest(2, 'link_growth_pct')]).drop_duplicates('station')
for _, r in labeled.iterrows():
    ax.annotate(r.station, (r.link_growth_pct, r.subway_growth_pct), xytext=(6, 4), textcoords='offset points',
                fontsize=11, color='#333333')
ax.text(0.98, 0.97, f"Spearman {t['통계량'].split(' (')[0]}, {t['p<0.05']}\n역 n = {t.n}",
        transform=ax.transAxes, ha='right', va='top', fontsize=14,
        bbox=dict(boxstyle='round', fc='white', ec='#cccccc'))
ax.set_xlabel('따릉이 역 연계 이동 증가율 (%)')
ax.set_ylabel('지하철 이용 증가율 (%)')
ax.set_title('따릉이 역 연계 이동 증가율 vs 지하철 증가율 (2019→2026, 5~6월 평일)')
style_axis(ax)
fig.tight_layout()
plt.show()

In [ ]:
# [11] 10_robustness_rho — 조건별 ①-1·②-2 Spearman ρ (본 분석 / (a) 공통 대여소만 / (b) 하루 값 중앙값 / (c) 상위 5% 역 제외)
conds = [('본 분석', '본 분석', '본 분석'), ('(a) 공통 대여소', '(a)', '(a) 두 해 공통\n대여소만'),
         ('(b) 하루 값 중앙값', '(b)', '(b) 하루 값\n중앙값'), ('(c) 상위 5% 역 제외', '(c)', '(c) 지하철 증가율\n상위 5% 역 제외')]
series = [('①-1', '①-1 역간 단거리 이동', '#1baf7a'), ('②-2', '②-2 역 연계 이동', '#eda100')]
fig, ax = plt.subplots(figsize=(13, 7))
width = 0.36
for i, (test, label, color) in enumerate(series):
    for j, (variant, pkey, _) in enumerate(conds):
        r = robust[(robust.test == test) & (robust.variant == variant)].iloc[0]
        p = robust_p[(robust_p.test == test) & (robust_p.variant == pkey)].p_value.iloc[0]
        rho = float(r.stat)
        n = r.n
        x = j + (i - 0.5) * width
        ax.bar(x, rho, width=width - 0.03, color=color, label=label if j == 0 else None)
        star = '**' if p < 0.01 else ('*' if p < 0.05 else '')
        ax.text(x, rho + 0.008, f'{rho:.2f}{star}\n(n={n})', ha='center', va='bottom', fontsize=12, color='#333333')
ax.axhline(0, color='#333333', lw=1.2)
ax.annotate('2026 이동 28%만 남음', xy=(1, 0.06), xytext=(1, 0.17), ha='center', fontsize=13, color='#b03a2e',
            arrowprops=dict(arrowstyle='->', color='#b03a2e', lw=1.2))
ax.set_xticks(range(len(conds)), [c[2] for c in conds])
ax.set_ylabel('Spearman ρ (따릉이 증가율 vs 지하철 증가율)')
ax.set_ylim(-0.05, 0.36)
ax.set_title('조건을 바꿔도 음(-)은 없다 — 공통 대여소만 보면 양(+)의 관계가 사라짐')
ax.text(0.99, 0.97, '*p<0.05, **p<0.01 / 별표 없음: p>0.05', transform=ax.transAxes, ha='right', va='top',
        fontsize=11, color='#666666')
style_axis(ax)
ax.legend(frameon=False, loc='upper left')
fig.tight_layout()
plt.show()

In [ ]:
# [12] 가설① 분포 — 역별 따릉이 역간 단거리 이동 증가율 / 지하철 증가율 히스토그램 (정규성 확인: Shapiro)
from scipy import stats


def normality_label(values):
    p = stats.shapiro(values).pvalue
    return f'n={len(values)}, ' + ('정규 아님 (p<0.05)' if p < 0.05 else '정규로 볼 수 있음 (p>0.05)')


fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (col, title, color) in zip(axes, [('hop_growth_pct', '따릉이 역간 단거리 이동 증가율', '#1baf7a'),
                                          ('subway_growth_pct', '지하철 이용 증가율', '#2a78d6')]):
    v = hyp1_growth[col]
    ax.hist(v, bins=40, color=color, edgecolor='white', lw=0.6)
    ax.axvline(v.median(), color='#333333', ls='--', lw=1.2)
    ax.text(0.97, 0.95, f'{normality_label(v)}\n중앙값 {v.median():+.1f}%', transform=ax.transAxes,
            ha='right', va='top', fontsize=13, bbox=dict(boxstyle='round', fc='white', ec='#cccccc'))
    ax.set_title(title)
    ax.set_xlabel('증가율 (%, 2019→2026 평일 하루 평균)')
    ax.set_ylabel('역 수')
    style_axis(ax)
fig.suptitle('가설① 검정 변수의 분포: 둘 다 오른쪽 꼬리가 긴 비정규 → Spearman·순위합 검정(W) 사용')
fig.tight_layout()
plt.show()

In [ ]:
# [13] 가설② 분포 — ②-1 평일 하루 7~9시 역 근처 반납 비중(2019·2026) / ②-2 역별 따릉이 역 연계 이동 증가율 히스토그램
hyp2_daily = pd.read_csv(TABLES / '04_hyp2_daily_share.csv')
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
bins = np.linspace(hyp2_daily.morning_share_pct.min(), hyp2_daily.morning_share_pct.max(), 25)
for year in YEARS:
    v = hyp2_daily.loc[hyp2_daily.year == year, 'morning_share_pct']
    ax.hist(v, bins=bins, color=COLOR[year], alpha=0.6, edgecolor='white', lw=0.6,
            label=f'{year}년: {normality_label(v)}')
ax.set_title('②-1 평일 하루 7~9시 역 근처 반납 비중')
ax.set_xlabel('역 근처 반납 중 7~9시 비중 (%)')
ax.set_ylabel('평일 수 (일)')
ax.legend(frameon=True, framealpha=0.9, loc='upper left', fontsize=12)
style_axis(ax)

ax = axes[1]
v = hyp2_growth.link_growth_pct
ax.hist(v, bins=40, color='#eda100', edgecolor='white', lw=0.6)
ax.axvline(v.median(), color='#333333', ls='--', lw=1.2)
ax.text(0.97, 0.95, f'{normality_label(v)}\n중앙값 {v.median():+.1f}%', transform=ax.transAxes,
        ha='right', va='top', fontsize=13, bbox=dict(boxstyle='round', fc='white', ec='#cccccc'))
ax.set_title('②-2 따릉이 역 연계 이동 증가율')
ax.set_xlabel('증가율 (%, 2019→2026 평일 하루 평균)')
ax.set_ylabel('역 수')
style_axis(ax)

fig.suptitle('가설② 검정 변수의 분포: 모두 비정규 → 순위합 검정(W)·Spearman 사용')
fig.tight_layout()
plt.show()